[矢吹太朗『Webのしくみ』（サイエンス社, 2020）](https://github.com/taroyabuki/webbook)

下のアイコンをクリックすれば，この文書に掲載されているコードを，Google Colab上で実行できます．（Googleのアカウントが必要です．）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taroyabuki/webbook/blob/master/chapters/08_cipher.ipynb)

# 第8章 暗号


## 8.1 事後否認・改ざん・なりすまし・盗聴

In [12]:
2+3


5

## 8.2 暗号の基本

![図8.1 シーザー暗号（「+13」文字ずらす）を使って，アリスがボブにメッセージを送っている様子](https://raw.githubusercontent.com/taroyabuki/webbook/master/chapters/figures/08-1.svg)

In [1]:
#準備
!apt-get install -y -qq hxtools > /dev/null

In [2]:
#暗号化
!echo "hello" | rot13

uryyb


In [3]:
#復号
!echo "hello" | rot13 | rot13

hello


## 8.3 公開鍵暗号

### 8.3.1 盗聴への対策

アリスはボブの公開鍵で暗号化し，ボブはボブの秘密鍵で復号します．

![図8.2 公開鍵暗号を使った暗号化と復号](https://raw.githubusercontent.com/taroyabuki/webbook/master/chapters/figures/08-2.svg)

ファイル|役割
--|--
bob-public.pem|ボブの公開鍵
bob-private.pem|ボブの秘密鍵
message4bob|ボブへのメッセージ
message4bob-secret|ボブへの暗号文


In [ ]:
#(1) ボブが公開鍵と秘密鍵を作る．
!openssl genrsa -out bob-private.pem 2048
!openssl rsa -in bob-private.pem -pubout -out bob-public.pem

In [5]:
#公開鍵を見てみる．
!cat bob-public.pem

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEApWuTR4Y8lA/AF/FFA17k
PpFgaWdPFjLeajJK1KFYb3gYOSQTKHtvTAt7XI2xgRxkFdGFw7nPp+ybwHs+5nNL
22UAttwgVKoJ0mEqELbUmsOImvTKcW+OSYKU3ojW+TYKGvbqsTESn71sNN9VGbws
o3IDLg/AcIFHPO6xXs88lMuJLcRD8YEDmuN2uxmYTQVDwHx0GiponBdzjrXFOEZY
/oPTzZYbk46OJWotzNphSLLs8JMkTsvCW9kh7tGw9zAzHD9Se4FOpCqsnMPFunT8
pp/xyhn9qUYQNRwyvSg9vZclGMlFutufPjVwCAecFr5Z4Q/35Ii4wg2rE2rZYt+l
OwIDAQAB
-----END PUBLIC KEY-----


In [ ]:
#秘密鍵を見てみる（秘密鍵は本来見せたり公開したりしてはいけない）．
!cat bob-private.pem

In [ ]:
#(2) ボブが公開鍵を公開する．

#(3) アリスがボブの公開鍵を手に入れる．

#(4) アリスがボブの公開鍵を使ってメッセージを暗号化し，ボブに送る．

#(4.1) メッセージをファイルに書く．
!printf "こんにちは\n" > message4bob

#(4.2) ボブの公開鍵を使って暗号文を作る．
!openssl pkeyutl -encrypt -pubin -inkey bob-public.pem -in message4bob -out message4bob-secret
!base64 message4bob-secret

In [8]:
#(5) ボブが暗号文を受け取り，秘密鍵を使って復号する．
!openssl pkeyutl -decrypt -inkey bob-private.pem -in message4bob-secret

こんにちは


### 8.3.2 デジタル署名

アリスはアリスの秘密鍵でデジタル署名を作り，ボブはアリスの公開鍵でデジタル署名を検証します．

ファイル|役割
--|--
alice-public.pem|アリスの公開鍵
alice-private.pem|アリスの秘密鍵
alice-message|アリスのメッセージ
alice-message-signature|アリスのデジタル署名

#### デジタル署名

In [ ]:
#(1) アリスが公開鍵と秘密鍵を作る．
!openssl genrsa -out alice-private.pem 2048
!openssl rsa -in alice-private.pem -pubout -out alice-public.pem

In [ ]:
#(2) アリスがメッセージをファイルに書き，アリスが秘密鍵を使ってデジタル署名を作る．
!printf "HELLO\n" > alice-message
!openssl dgst -sha256 -sign alice-private.pem -out alice-message-signature alice-message

In [11]:
#(3) アリスが公開鍵を公開する．

#(4) ボブがアリスの公開鍵，メッセージ，デジタル署名を取得する．

#(5) ボブがアリスの公開鍵を使って，メッセージのデジタル署名を検証する．
!openssl dgst -sha256 -verify alice-public.pem -signature alice-message-signature alice-message

Verified OK


### 8.3.3 デジタル証明書

## 8.4 ウェブの安全性

### 8.4.1 TLS